 # QuantRisk AI — Module 02
## Exploratory Data Analysis (EDA)

### Objective

The objective of this module is to explore the historical behavior of
NIFTY stocks and understand their return, volatility, risk, and relationships
with broader market indicators such as NIFTY 50 and India VIX.

The insights obtained from EDA will guide feature engineering and the
development of machine-learning and deep-learning risk models.

In [ ]:
# ==========================================
# Import Required Libraries
# ==========================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# ==========================================
# Load Master Dataset
# ==========================================

file_path = "master_dataset.csv"

df = pd.read_csv(file_path)

df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df["Date"]=pd.to_datetime(df["Date"])
df.dtypes

In [ ]:
print("Dataset Shape:", df.shape)

print(
    "Number of Stocks:",
    df["Stock"].nunique()
)

print(
    "Date Range:",
    df["Date"].min(),
    "to",
    df["Date"].max()
)

print(
    "Total Missing Values:",
    df.isnull().sum().sum()
)

print(
    "Duplicate Stock-Date Rows:",
    df.duplicated(
        subset=["Date", "Stock"]
    ).sum()
)

In [ ]:
df.describe()

In [ ]:
# ==========================================
# Sort Data for Time-Series Calculations
# ==========================================

df = df.sort_values(
    by=["Stock", "Date"]
).reset_index(drop=True)

In [ ]:
# ==========================================
# Calculate Daily Stock Returns
# ==========================================

df["Daily_Return"] = (
    df.groupby("Stock")["Close"]
      .pct_change(fill_method=None)
)

df[
    ["Date", "Stock", "Close", "Daily_Return"]
].head(500)

In [ ]:
print(
    "Missing Daily Returns:",
    df["Daily_Return"].isnull().sum()
)

In [ ]:
df["Daily_Return"].describe()

In [ ]:
# ==========================================
# Largest Daily Loss
# ==========================================

min_return_row = df.loc[
    df["Daily_Return"].idxmin(),
    ["Date", "Stock", "Close", "Daily_Return"]
]

min_return_row

In [ ]:
# ==========================================
# Largest Daily Gain
# ==========================================

max_return_row = df.loc[
    df["Daily_Return"].idxmax(),
    ["Date", "Stock", "Close", "Daily_Return"]
]

max_return_row

In [ ]:
# ==========================================
# Daily Return Percentage
# ==========================================

df["Daily_Return_Pct"] = (
    df["Daily_Return"] * 100
)

In [ ]:
df[
    [
        "Date",
        "Stock",
        "Close",
        "Daily_Return",
        "Daily_Return_Pct"
    ]
].head()

In [ ]:
# ==========================================
# Stock-wise Return Statistics
# ==========================================

stock_return_stats = (
    df.groupby("Stock")["Daily_Return"]
      .agg(
          Mean_Daily_Return="mean",
          Daily_Volatility="std",
          Min_Return="min",
          Max_Return="max"
      )
      .reset_index()
)

stock_return_stats.head(10)

In [ ]:
stock_return_stats[
    [
        "Mean_Daily_Return",
        "Daily_Volatility",
        "Min_Return",
        "Max_Return"
    ]
] *= 100

In [ ]:
stock_return_stats.head(10)

In [ ]:
# ==========================================
# Top 10 Stocks by Mean Daily Return
# ==========================================

top_return_stocks = (
    stock_return_stats
    .sort_values(
        "Mean_Daily_Return",
        ascending=False
    )
    .head(10)
)

top_return_stocks[
    [
        "Stock",
        "Mean_Daily_Return",
        "Daily_Volatility"
    ]
]

In [ ]:
# ==========================================
# Top 10 Stocks by Daily Volatility
# ==========================================

top_volatile_stocks = (
    stock_return_stats
    .sort_values(
        "Daily_Volatility",
        ascending=False
    )
    .head(10)
)

top_volatile_stocks[
    [
        "Stock",
        "Daily_Volatility",
        "Mean_Daily_Return"
    ]
]

In [ ]:
# ==========================================
# Top 10 Lowest-Volatility Stocks
# ==========================================

low_volatile_stocks = (
    stock_return_stats
    .sort_values(
        "Daily_Volatility",
        ascending=True
    )
    .head(10)
)

low_volatile_stocks[
    [
        "Stock",
        "Daily_Volatility",
        "Mean_Daily_Return"
    ]
]

In [ ]:
# ==========================================
# Risk vs Return Scatter Plot
# ==========================================

plt.figure(figsize=(12, 8))

plt.scatter(
    stock_return_stats["Daily_Volatility"],
    stock_return_stats["Mean_Daily_Return"],
    alpha=0.7
)

# Add stock labels
for _, row in stock_return_stats.iterrows():
    plt.annotate(
        row["Stock"],
        (
            row["Daily_Volatility"],
            row["Mean_Daily_Return"]
        ),
        fontsize=7,
        alpha=0.8
    )

plt.xlabel("Daily Volatility (%)")
plt.ylabel("Mean Daily Return (%)")

plt.title(
    "Historical Risk vs Return of NIFTY Stocks"
)

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# Simple Return-to-Volatility Ratio
# ==========================================

stock_return_stats["Return_Volatility_Ratio"] = (
    stock_return_stats["Mean_Daily_Return"]
    /
    stock_return_stats["Daily_Volatility"]
)

risk_adjusted_ranking = (
    stock_return_stats
    .sort_values(
        "Return_Volatility_Ratio",
        ascending=False
    )
)

risk_adjusted_ranking[
    [
        "Stock",
        "Mean_Daily_Return",
        "Daily_Volatility",
        "Return_Volatility_Ratio"
    ]
].head(10)

In [ ]:
# ==========================================
# Distribution of Daily Stock Returns
# ==========================================

plt.figure(figsize=(12, 6))

plt.hist(
    df["Daily_Return_Pct"].dropna(),
    bins=100
)

plt.xlabel("Daily Return (%)")
plt.ylabel("Frequency")

plt.title(
    "Distribution of Daily Returns Across NIFTY Stocks"
)

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# Distribution of Typical Daily Returns
# ==========================================

typical_returns = df[
    df["Daily_Return_Pct"].between(-10, 10)
]["Daily_Return_Pct"]

plt.figure(figsize=(12, 6))

plt.hist(
    typical_returns.dropna(),
    bins=100
)

plt.xlabel("Daily Return (%)")
plt.ylabel("Frequency")

plt.title(
    "Distribution of Daily Returns (-10% to +10%)"
)

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# Extreme Daily Return Analysis
# ==========================================

large_positive = (
    df["Daily_Return_Pct"] > 5
).sum()

large_negative = (
    df["Daily_Return_Pct"] < -5
).sum()

total_returns = (
    df["Daily_Return_Pct"]
    .notna()
    .sum()
)

print("Returns greater than +5%:", large_positive)
print("Returns less than -5%:", large_negative)

print(
    "\nPercentage > +5%:",
    round(
        large_positive / total_returns * 100,
        2
    ),
    "%"
)

print(
    "Percentage < -5%:",
    round(
        large_negative / total_returns * 100,
        2
    ),
    "%"
)

In [ ]:
# ==========================================
# Skewness and Kurtosis of Daily Returns
# ==========================================

return_skewness = (
    df["Daily_Return"]
    .dropna()
    .skew()
)

return_kurtosis = (
    df["Daily_Return"]
    .dropna()
    .kurt()
)

print(
    "Skewness:",
    round(return_skewness, 4)
)

print(
    "Kurtosis:",
    round(return_kurtosis, 4)
)

### Daily Return Distribution Insights

The distribution of pooled daily stock returns is highly concentrated
around zero, indicating that most daily price movements are relatively
small.

Approximately 1.51% of observations recorded returns greater than +5%,
while 1.19% recorded returns below -5%. Overall, approximately 2.70% of
valid daily returns were outside the ±5% range.

The return distribution has a skewness of -0.0342, indicating that the
overall distribution is approximately symmetric.

However, the excess kurtosis is 16.0422, which is substantially higher
than that of a normal distribution. This indicates heavy tails and the
presence of extreme return observations.

These findings suggest that volatility alone may not fully represent
financial risk, motivating the later use of additional downside and
tail-risk measures.

In [ ]:
df["Daily_Return"]

In [ ]:
# ==========================================
# Step 24: 20-Day Rolling Volatility
# ==========================================

df["Rolling_Volatility_20D"] = (
    df.groupby("Stock")["Daily_Return"]
      .rolling(window=20)
      .std()
      .reset_index(level=0, drop=True)
)

df[
    [
        "Date",
        "Stock",
        "Daily_Return",
        "Rolling_Volatility_20D"
    ]
].head(25)

In [ ]:
import matplotlib.pyplot as plt

adanient = df[df["Stock"] == "ADANIENT"].copy()

plt.figure(figsize=(14, 5))

plt.plot(
    adanient["Date"],
    adanient["Rolling_Volatility_20D"] * 100
)

plt.title("ADANIENT - 20-Day Rolling Volatility")
plt.xlabel("Date")
plt.ylabel("Rolling Volatility (%)")
plt.grid(True)

plt.show()

In [ ]:
# ==========================================
# Step 26: Correct NIFTY 50 Daily Return
# ==========================================

nifty = (
    df[["Date", "NIFTY_Close"]]
    .drop_duplicates("Date")
    .sort_values("Date")
    .copy()
)

nifty["NIFTY_Return"] = nifty["NIFTY_Close"].pct_change()

nifty.head(10)

In [ ]:
# ==========================================
# Step 27: NIFTY 50 Risk Statistics
# ==========================================

nifty["NIFTY_Return"].describe()

In [ ]:
# ==========================================
# Step 28: Annualized NIFTY Volatility
# ==========================================

nifty_daily_vol = nifty["NIFTY_Return"].std()

nifty_annual_vol = nifty_daily_vol * (252 ** 0.5)

print("NIFTY Daily Volatility:", round(nifty_daily_vol * 100, 2), "%")
print("NIFTY Annualized Volatility:", round(nifty_annual_vol * 100, 2), "%")

In [ ]:
# ==========================================
# Step 29: India VIX Statistics
# ==========================================

vix = (
    df[["Date", "VIX_Close"]]
    .drop_duplicates("Date")
    .sort_values("Date")
    .copy()
)

vix["VIX_Return"] = vix["VIX_Close"].pct_change()

print("VIX Descriptive Statistics:")
print(vix["VIX_Close"].describe())

In [ ]:
# ==========================================
# Step 30: Highest India VIX Observations
# ==========================================

top_vix = (
    vix[["Date", "VIX_Close"]]
    .sort_values("VIX_Close", ascending=False)
    .head(10)
)

top_vix

In [ ]:
# ==========================================
# Step 31: India VIX Time Series
# ==========================================

import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.plot(
    vix["Date"],
    vix["VIX_Close"]
)

plt.title("India VIX - Historical Volatility Regime")
plt.xlabel("Date")
plt.ylabel("India VIX")
plt.grid(True)

plt.show()

In [ ]:
# ==========================================
# Step 32: NIFTY Return vs VIX Change
# ==========================================

market = nifty.merge(
    vix[["Date", "VIX_Close"]],
    on="Date",
    how="inner"
)

market["VIX_Return"] = market["VIX_Close"].pct_change()

market[["Date", "NIFTY_Return", "VIX_Close", "VIX_Return"]].head(10)

In [ ]:
correlation = market[
    ["NIFTY_Return", "VIX_Return"]
].corr()

print(correlation)

In [ ]:
# ==========================================
# Step 33: Stock Returns vs NIFTY Returns
# ==========================================

stock_market = df[
    ["Date", "Stock", "Daily_Return"]
].merge(
    nifty[["Date", "NIFTY_Return"]],
    on="Date",
    how="inner"
)

stock_market.head()

In [ ]:
# ==========================================
# Step 34: Stock-NIFTY Correlation
# ==========================================

stock_nifty_corr = (
    stock_market
    .groupby("Stock")
    .apply(
        lambda x: x["Daily_Return"].corr(x["NIFTY_Return"]),
        include_groups=False
    )
    .sort_values(ascending=False)
)

stock_nifty_corr.head(10)

In [ ]:
stock_nifty_corr.tail(10)

In [ ]:
# ==========================================
# Step 36: Stock Drawdown
# ==========================================

df["Running_Max"] = (
    df.groupby("Stock")["Close"]
      .cummax()
)

df["Drawdown"] = (
    df["Close"] / df["Running_Max"]
) - 1

df[[
    "Date",
    "Stock",
    "Close",
    "Running_Max",
    "Drawdown"
]].head(25)

In [ ]:
# ==========================================
# Step 37: Maximum Drawdown by Stock
# ==========================================

max_drawdown = (
    df.groupby("Stock")["Drawdown"]
      .min()
      .sort_values()
)

print(max_drawdown.head(10))

# Module 3 - Feature engineering

In [ ]:
# ==========================================
# MODULE 3 - STEP 3
# 60-Day Rolling Volatility
# ==========================================

df["Rolling_Volatility_60D"] = (
    df.groupby("Stock")["Daily_Return"]
      .transform(lambda x: x.rolling(window=60).std())
)

df[[
    "Date",
    "Stock",
    "Daily_Return",
    "Rolling_Volatility_20D",
    "Rolling_Volatility_60D"
]].head(65)

In [ ]:
# ==========================================
# MODULE 3 - STEP 4
# 20-Day Moving Average
# ==========================================

df["MA_20"] = (
    df.groupby("Stock")["Close"]
      .transform(lambda x: x.rolling(window=20).mean())
)

df[[
    "Date",
    "Stock",
    "Close",
    "MA_20"
]].head(25)

In [ ]:
# ==========================================
# MODULE 3 - STEP 5
# 50-Day Moving Average
# ==========================================

df["MA_50"] = (
    df.groupby("Stock")["Close"]
      .transform(lambda x: x.rolling(window=50).mean())
)

df[[
    "Date",
    "Stock",
    "Close",
    "MA_20",
    "MA_50"
]].head(55)

In [ ]:
# ==========================================
# MODULE 3 - STEP 6
# Price-to-Moving-Average Features
# ==========================================

df["Price_MA20_Ratio"] = (
    df["Close"] / df["MA_20"]
)

df["Price_MA50_Ratio"] = (
    df["Close"] / df["MA_50"]
)

df[[
    "Date",
    "Stock",
    "Close",
    "MA_20",
    "MA_50",
    "Price_MA20_Ratio",
    "Price_MA50_Ratio"
]].iloc[45:55]

In [ ]:
# ==========================================
# MODULE 3 - STEP 7
# 20-Day Momentum
# ==========================================

df["Momentum_20D"] = (
    df.groupby("Stock")["Close"]
      .transform(lambda x: x.pct_change(periods=20))
)

df[[
    "Date",
    "Stock",
    "Close",
    "Momentum_20D"
]].iloc[18:25]

In [ ]:
# ==========================================
# MODULE 3 - STEP 8
# 14-Day RSI
# ==========================================

def calculate_rsi(series, period=14):
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))

    return rsi


df["RSI_14"] = (
    df.groupby("Stock")["Close"]
      .transform(calculate_rsi)
)

df[[
    "Date",
    "Stock",
    "Close",
    "RSI_14"
]].iloc[10:25]

In [ ]:
# ==========================================
# MODULE 3 - STEP 9
# MACD
# ==========================================

df["EMA_12"] = (
    df.groupby("Stock")["Close"]
      .transform(lambda x: x.ewm(span=12, adjust=False).mean())
)

df["EMA_26"] = (
    df.groupby("Stock")["Close"]
      .transform(lambda x: x.ewm(span=26, adjust=False).mean())
)

df["MACD"] = df["EMA_12"] - df["EMA_26"]

df["MACD_Signal"] = (
    df.groupby("Stock")["MACD"]
      .transform(lambda x: x.ewm(span=9, adjust=False).mean())
)

df["MACD_Histogram"] = (
    df["MACD"] - df["MACD_Signal"]
)

In [ ]:
df[[
    "Date",
    "Stock",
    "Close",
    "EMA_12",
    "EMA_26",
    "MACD",
    "MACD_Signal",
    "MACD_Histogram"
]].head(20)

In [ ]:
df[[
    "Date",
    "MACD",
    "MACD_Signal",
    "MACD_Histogram"
]].head(5)

In [ ]:
print(df.loc[0, "MACD"])
print(df.loc[1, "MACD"])
print(df.loc[1, "MACD_Signal"])

print(df.groupby("Stock")["MACD"].head(3))

In [ ]:
# ==========================================
# MODULE 3 - STEP 10A
# TRUE RANGE (TR)
# ==========================================

df["Previous_Close"] = (
    df.groupby("Stock")["Close"].shift(1)
)

df["TR"] = df[
    ["High", "Low", "Previous_Close"]
].apply(
    lambda row: max(
        row["High"] - row["Low"],
        abs(row["High"] - row["Previous_Close"]),
        abs(row["Low"] - row["Previous_Close"])
    ),
    axis=1
)

In [ ]:
df[[
    "Date",
    "Stock",
    "High",
    "Low",
    "Previous_Close",
    "TR"
]].head(20)

In [ ]:
# ==========================================
# MODULE 3 - STEP 10B
# ATR (14-DAY)
# ==========================================

df["ATR_14"] = (
    df.groupby("Stock")["TR"]
      .transform(lambda x: x.rolling(window=14).mean())
)

In [ ]:
df[[
    "Date",
    "Stock",
    "TR",
    "ATR_14"
]].head(25)

In [ ]:
# ==========================================
# MODULE 3 - STEP 11
# BETA
# ==========================================

stock_market = df[
    ["Date", "Stock", "Daily_Return"]
].merge(
    nifty[["Date", "NIFTY_Return"]],
    on="Date",
    how="inner"
)

In [ ]:
beta_by_stock = (
    stock_market.groupby("Stock")
    .apply(
        lambda x: (
            x["Daily_Return"].cov(x["NIFTY_Return"])
            / x["NIFTY_Return"].var()
        ),
        include_groups=False
    )
    .sort_values(ascending=False)
)

beta_by_stock.head(10)

In [ ]:
beta_by_stock.tail(10)

In [ ]:
df["Beta"] = df["Stock"].map(beta_by_stock)

In [ ]:
df[[
    "Date",
    "Stock",
    "Daily_Return",
    "Beta"
]].head(20)

In [ ]:
# ==========================================
# MODULE 3 - STEP 12
# VOLATILITY RATIO
# ==========================================

df["Volatility_Ratio"] = (
    df["Rolling_Volatility_20D"]
    / df["Rolling_Volatility_60D"]
)

In [ ]:
df[[
    "Date",
    "Stock",
    "Rolling_Volatility_20D",
    "Rolling_Volatility_60D",
    "Volatility_Ratio"
]].head(70)

In [ ]:
# ==========================================
# MODULE 3 - STEP 13A
# VOLUME RATIO
# ==========================================

df["Volume_MA20"] = (
    df.groupby("Stock")["Volume"]
      .transform(lambda x: x.rolling(window=20).mean())
)

df["Volume_Ratio"] = (
    df["Volume"] / df["Volume_MA20"]
)

In [ ]:
df[[
    "Date",
    "Stock",
    "Volume",
    "Volume_MA20",
    "Volume_Ratio"
]].head(25)

In [ ]:
# ==========================================
# MODULE 3 - STEP 13B
# DAILY VOLUME CHANGE
# ==========================================

df["Volume_Change"] = (
    df.groupby("Stock")["Volume"]
      .transform(lambda x: x.pct_change())
)

In [ ]:
df[[
    "Date",
    "Stock",
    "Volume",
    "Volume_Change"
]].head(25)

In [ ]:
# ==========================================
# MODULE 3 - STEP 14
# INTRADAY PRICE RANGE
# ==========================================

df["Price_Range"] = (
    df["High"] - df["Low"]
)

In [ ]:
df[[
    "Date",
    "Stock",
    "High",
    "Low",
    "Price_Range"
]].head(20)

In [ ]:
# ==========================================
# MODULE 3 - STEP 15
# HIGH-LOW PERCENTAGE
# ==========================================

df["High_Low_Pct"] = (
    (df["High"] - df["Low"]) / df["Close"]
)

In [ ]:
df[[
    "Date",
    "Stock",
    "High",
    "Low",
    "Close",
    "High_Low_Pct"
]].head(20)

In [ ]:
# ==========================================
# MODULE 3 - STEP 16
# GAP PERCENTAGE
# ==========================================

df["Gap_Pct"] = (
    (df["Open"] - df["Previous_Close"])
    / df["Previous_Close"]
)

In [ ]:
df[[
    "Date",
    "Stock",
    "Previous_Close",
    "Open",
    "Gap_Pct"
]].head(20)

In [ ]:
# ==========================================
# MODULE 3 - STEP 17
# FEATURE QUALITY CHECK
# ==========================================

print("Shape:", df.shape)

print("\nNaN count:")
print(df.isna().sum())

print("\nInfinite values:")
print(np.isinf(df.select_dtypes(include=np.number)).sum())

print("\nConstant numerical columns:")
print(
    df.select_dtypes(include=np.number).nunique()
    [df.select_dtypes(include=np.number).nunique() <= 1]
)

In [ ]:
print("\nDuplicate Stock-Date combinations:")
print(
    df.duplicated(subset=["Stock", "Date"]).sum()
)

In [ ]:
# ==========================================
# INVESTIGATE INFINITE VOLUME CHANGE
# ==========================================

inf_volume = df[
    np.isinf(df["Volume_Change"])
][[
    "Date",
    "Stock",
    "Volume",
    "Volume_Change"
]]

print("Number of infinite values:", len(inf_volume))

inf_volume.head(20)

In [ ]:
df[
    df["Volume_Change"].isin([np.inf, -np.inf])
][["Stock", "Volume"]].head(20)

In [ ]:
# ==========================================
# FIX INFINITE VOLUME CHANGE VALUES
# ==========================================

df["Volume_Change"] = df["Volume_Change"].replace(
    [np.inf, -np.inf],
    np.nan
)

In [ ]:
print("Infinite Volume_Change values:")
print(np.isinf(df["Volume_Change"]).sum())

print("\nNaN Volume_Change values:")
print(df["Volume_Change"].isna().sum())

In [ ]:
print(
    df["Volume_Change"]
    .isna()
    .sum()
)

print(
    np.isinf(df["Volume_Change"])
    .sum()
)

In [ ]:
# Check all NaN Volume_Change rows
volume_nan = df[
    df["Volume_Change"].isna()
][[
    "Date",
    "Stock",
    "Volume"
]]

print("Total NaN:", len(volume_nan))
volume_nan.tail(15)

In [ ]:
# Check NaN Volume_Change where the stock has a previous row
check = df.copy()

check["Row_In_Stock"] = (
    check.groupby("Stock").cumcount()
)

print(
    check[
        (check["Volume_Change"].isna()) &
        (check["Row_In_Stock"] > 0)
    ][[
        "Date",
        "Stock",
        "Volume",
        "Row_In_Stock"
    ]]
)

In [ ]:
# ==========================================
# MODULE 3 - FINAL QUALITY CHECK
# ==========================================

numeric_cols = df.select_dtypes(include=np.number).columns

print("Total infinite values:")
print(
    np.isinf(df[numeric_cols]).sum().sum()
)

print("\nTotal NaN values:")
print(
    df[numeric_cols].isna().sum().sum()
)

print("\nDuplicate Stock-Date rows:")
print(
    df.duplicated(subset=["Stock", "Date"]).sum()
)

In [ ]:
# ==========================================
# MODULE 3 - FINAL CORRELATION ANALYSIS
# ==========================================

feature_cols = [
    "Daily_Return",
    "Rolling_Volatility_20D",
    "Rolling_Volatility_60D",
    "MA_20",
    "MA_50",
    "Price_MA20_Ratio",
    "Price_MA50_Ratio",
    "Momentum_20D",
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "MACD_Histogram",
    "ATR_14",
    "Beta",
    "Volatility_Ratio",
    "Volume_MA20",
    "Volume_Ratio",
    "Volume_Change",
    "Price_Range",
    "High_Low_Pct",
    "Gap_Pct"
]

corr_matrix = df[feature_cols].corr()

corr_matrix.round(2)

In [ ]:
corr_pairs = (
    corr_matrix
    .where(
        np.triu(
            np.ones(corr_matrix.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .sort_values(ascending=False)
)

corr_pairs.head(15)

In [ ]:
corr_pairs.sort_values().head(15)

In [ ]:
# ============================================================
# Final Candidate Features After Correlation Analysis
# ============================================================

selected_features = [
    "Daily_Return",
    "Rolling_Volatility_20D",
    "Rolling_Volatility_60D",
    "Price_MA20_Ratio",
    "Price_MA50_Ratio",
    "Momentum_20D",
    "RSI_14",
    "MACD",
    "MACD_Histogram",
    "ATR_14",
    "Beta",
    "Volatility_Ratio",
    "Volume_MA20",
    "Volume_Ratio",
    "Volume_Change",
    "Price_Range",
    "High_Low_Pct",
    "Gap_Pct"
]

print("Number of candidate features:", len(selected_features))

print("\nCandidate features:")
for i, feature in enumerate(selected_features, start=1):
    print(f"{i}. {feature}")